In [ ]:
import pandas as pd
import sqlite3


In [ ]:
daily = pd.read_csv('../01_Data/processed/daily_energy.csv')


In [ ]:
hourly = pd.read_csv('../01_Data/processed/hourly_energy.csv')


In [ ]:
daily.head()


## SQL data

In [ ]:
con = sqlite3.connect('../03_Analytics_BI_SQL/sql/household_energy.db')


In [ ]:
daily.to_sql("daily_energy", con, if_exists="replace", index=False)


In [ ]:
hourly.to_sql("hourly_energy", con, if_exists="replace", index=False)


## Average energy by day of week

In [ ]:
query = """SELECT dayofweek, AVG(daily_kwh) AS avg_kwh FROM daily_energy GROUP BY dayofweek ORDER BY dayofweek"""


In [ ]:
pd.read_sql(query, con)


## Average energy by month

In [ ]:
query = """SELECT month, AVG(daily_kwh) AS avg_kwh FROM daily_energy GROUP BY month ORDER BY month"""


In [ ]:
monthly = pd.read_sql(query, con)


In [ ]:
monthly.head()


## Excel summary

In [ ]:
weekday = daily[daily["is_weekend"] == 0]["daily_kwh"].mean()


In [ ]:
weekend = daily[daily["is_weekend"] == 1]["daily_kwh"].mean()


In [ ]:
uplift = (weekend / weekday - 1) * 100


In [ ]:
kpi = pd.DataFrame({"Metric":["Avg daily kWh","Weekend uplift","Usable days"], "Value":[daily["daily_kwh"].mean(), uplift, len(daily)]})


In [ ]:
kpi


In [ ]:
kpi.to_excel('../03_Analytics_BI_SQL/excel/energy_kpi_summary.xlsx', index=False)


## Power BI files

In [ ]:
daily[['date','daily_kwh','is_weekend','month','dayofweek']].to_csv('../03_Analytics_BI_SQL/powerbi/powerbi_daily.csv', index=False)


In [ ]:
hourly[['date_time','energy_kwh','hour','is_weekend']].to_csv('../03_Analytics_BI_SQL/powerbi/powerbi_hourly.csv', index=False)


In [ ]:
con.close()
